# 02 — Module A: NLP Sentiment Classification
## Universal Sequence Lab · Assignment 3

**Task:** Binary sentiment classification on IMDb movie reviews (positive / negative).

**Models compared:**
1. Vanilla RNN (baseline — demonstrates vanishing gradient)
2. Bidirectional LSTM
3. Bidirectional GRU
4. Transformer (Encoder + mean pooling)

---

In [ ]:
!pip install datasets seaborn scikit-learn -q

import sys
sys.path.append('/content/src')  # adjust if running locally

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from models import VanillaRNN, LSTMClassifier, GRUClassifier, TransformerClassifier
from dataset import get_imdb_loaders
from train import train_model, plot_learning_curves, plot_confusion_matrix, \
                  print_classification_report, count_parameters, evaluate

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 1. Load Data


In [ ]:
BATCH_SIZE = 64
MAX_LEN    = 256

train_loader, val_loader, test_loader, vocab = get_imdb_loaders(
    batch_size=BATCH_SIZE, max_len=MAX_LEN)

VOCAB_SIZE  = len(vocab)
PAD_IDX     = vocab['<pad>']
N_CLASSES   = 2
print(f'Vocab size: {VOCAB_SIZE:,} | Pad idx: {PAD_IDX}')

## 2. Define Hyperparameters & Initialize Models


In [ ]:
# Shared hyperparameters
EMBED_DIM  = 128
HIDDEN_DIM = 256
N_LAYERS   = 2
DROPOUT    = 0.3
N_EPOCHS   = 10
LR         = 1e-3

# Transformer-specific
N_HEADS  = 4
FF_DIM   = 512
T_LAYERS = 3

models_config = {
    'VanillaRNN':  VanillaRNN(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_CLASSES,
                               N_LAYERS, DROPOUT, PAD_IDX),
    'BiLSTM':      LSTMClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_CLASSES,
                                   N_LAYERS, True, DROPOUT, PAD_IDX),
    'BiGRU':       GRUClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, N_CLASSES,
                                  N_LAYERS, True, DROPOUT, PAD_IDX),
    'Transformer': TransformerClassifier(VOCAB_SIZE, EMBED_DIM, N_HEADS, FF_DIM,
                                          N_CLASSES, T_LAYERS, DROPOUT, PAD_IDX),
}

print('Model parameter counts:')
for name, model in models_config.items():
    print(f'\n  {name}:')
    count_parameters(model)

## 3. Train All Models


In [ ]:
histories = {}
criterion = nn.CrossEntropyLoss()

for name, model in models_config.items():
    print(f'\n{'='*50}')
    print(f' Training: {name}')
    print(f'{'='*50}')
    model = model.to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

    hist = train_model(
        model, train_loader, val_loader, optimizer, criterion, DEVICE,
        n_epochs=N_EPOCHS, task='classification', use_lengths=True,
        scheduler=scheduler, model_name=name,
        save_path=f'/content/models/nlp_{name}.pt'
    )
    histories[name] = hist
    models_config[name] = model  # keep model in GPU

## 4. Learning Curves


In [ ]:
fig = plot_learning_curves(histories, task='classification')

## 5. Test Set Evaluation & Confusion Matrices


In [ ]:
results = {}

for name, model in models_config.items():
    # Load best checkpoint
    model.load_state_dict(torch.load(f'/content/models/nlp_{name}.pt',
                                      map_location=DEVICE))
    test_loss, preds, labels = evaluate(model, test_loader, criterion, DEVICE,
                                         task='classification', use_lengths=True)
    acc = (preds == labels).mean()
    results[name] = {'test_loss': test_loss, 'test_acc': acc,
                     'preds': preds, 'labels': labels}
    print(f'{name:15s} | Test Loss: {test_loss:.4f} | Test Acc: {acc:.4f}')

In [ ]:
for name, res in results.items():
    plot_confusion_matrix(res['labels'], res['preds'],
                          ['Negative', 'Positive'], title=f'{name} — Confusion Matrix')
    print_classification_report(res['labels'], res['preds'], ['Negative', 'Positive'])
    print()

## 6. Vanishing Gradient Analysis — RNN vs LSTM


In [ ]:
# Compare gradient norms at first vs last layer to illustrate vanishing gradient
def get_gradient_norms(model, loader, criterion, device, use_lengths=True):
    model.train()
    batch = next(iter(loader))
    src, lengths, labels = batch
    src, labels = src.to(device), labels.to(device)
    pred = model(src, lengths) if use_lengths else model(src)
    loss = criterion(pred, labels)
    loss.backward()
    norms = {}
    for name, param in model.named_parameters():
        if param.grad is not None:
            norms[name] = param.grad.norm().item()
    model.zero_grad()
    return norms

rnn_norms  = get_gradient_norms(models_config['VanillaRNN'], train_loader, criterion, DEVICE)
lstm_norms = get_gradient_norms(models_config['BiLSTM'],     train_loader, criterion, DEVICE)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, norms, title, color in zip(
    axes, [rnn_norms, lstm_norms], ['VanillaRNN Gradient Norms', 'BiLSTM Gradient Norms'],
    ['#e74c3c', '#2ecc71']):
    layers = list(norms.keys())
    values = list(norms.values())
    ax.barh(layers[-10:], values[-10:], color=color)  # last 10 params
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Gradient Norm')

plt.tight_layout()
plt.savefig('gradient_analysis.png', bbox_inches='tight')
plt.show()
print('Note: RNN gradients in early layers are near-zero (vanishing gradient problem)')

## 7. Module A Summary

| Model | Test Accuracy | Notes |
|-------|-------------|-------|
| VanillaRNN | ~0.78 | Struggles with long reviews (vanishing gradient) |
| BiLSTM | ~0.88 | Strong performance, captures long-range deps |
| BiGRU | ~0.88 | Comparable to LSTM, fewer parameters |
| Transformer | ~0.90 | Best accuracy; global attention bypasses sequence length issues |

> Numbers above are approximate; actual results filled in after training.
